# Pipeline Changes on a T4

Measures each pipeline change from
[docs/daily-risk-map.md](../docs/daily-risk-map.md#pipeline-changes)
against two runs with the default settings. The changes and their
success criteria were written before this notebook was run.

Run on Kaggle with **Accelerator: GPU T4 x2** and **Internet: on**. The
notebook uses one GPU and takes about 25 minutes.

The notebook gets the project from GitHub. To use a local copy instead,
upload the repository as a Kaggle dataset named `earth-intelligence`
and add it to the notebook.

Output: `/kaggle/working/pipeline_changes.zip`. Unzip it into
`ml/artifacts/pipeline_changes/`, then run
`scripts/map/compare_daily_runs.py`.

In [ ]:
!pip install -q "makani @ git+https://github.com/NVIDIA/makani.git@b38fcb2799d7dbc146fa60459f3f9823394a8bf1" "earth2studio[data,sfno]==0.20.0" python-dotenv ninja 2>&1 | tail -3

In [ ]:
import shutil
import subprocess
from pathlib import Path

REPOSITORY = "https://github.com/khoahonguyen0210/Earth_intelligence.git"
INPUT = Path("/kaggle/input")
PROJECT = Path("/kaggle/working/earth-intelligence")
OUTPUT = Path("/kaggle/working/pipeline_changes")


def uploaded_project():
    for marker in INPUT.rglob("ml/pyproject.toml"):
        return marker.parent.parent
    for archive in INPUT.rglob("earth-intelligence.tar.gz"):
        shutil.unpack_archive(archive, PROJECT)
        return PROJECT
    return None


if not PROJECT.exists():
    source = uploaded_project()
    if source is None:
        subprocess.run(
            ["git", "clone", "--depth", "1", REPOSITORY, str(PROJECT)], check=True
        )
    elif source != PROJECT:
        shutil.copytree(source, PROJECT)

## Download the model once

The first load downloads the SFNO package. Loading it here keeps the
download out of the timed runs.

In [ ]:
!python -c "from earth2studio.models.px import SFNO; SFNO.load_model(SFNO.load_default_package())"

## Runs

All runs use the same cutoff (2025-08-03 10:00 UTC), ERA5 initial
conditions from ARCO and eager PyTorch for the index. Each run starts a
new process and loads the model again. Each change runs alone, so its
effect is not mixed with the others.

In [ ]:
import json

CUTOFF = "2025-08-03T10:00:00+00:00"
RUNS = [
    ("baseline_1", ""),
    ("overlap_loading", "--overlap-loading"),
    ("cudnn_benchmark", "--cudnn-benchmark"),
    ("half_precision", "--half-precision"),
    ("larger_blocks", "--rows-per-chunk 128 --channels-per-chunk 64"),
    ("baseline_2", ""),
]
results = {}
for name, options in RUNS:
    command = (
        f"cd {PROJECT} && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True PYTHONPATH=ml/src "
        f"python scripts/map/run_daily_forecast.py --cutoff {CUTOFF} --source arco "
        f"--output {OUTPUT / name} {options}"
    )
    finished = subprocess.run(command, shell=True, capture_output=True, text=True)
    results[name] = {
        "return_code": finished.returncode,
        "last_lines": (finished.stdout + finished.stderr).splitlines()[-15:],
    }
    print(name, finished.returncode)

OUTPUT.mkdir(parents=True, exist_ok=True)
(OUTPUT / "runs.json").write_text(json.dumps(results, indent=2))

In [ ]:
!cd {PROJECT} && PYTHONPATH=ml/src python scripts/map/compare_daily_runs.py {OUTPUT}

In [ ]:
shutil.make_archive("/kaggle/working/pipeline_changes", "zip", OUTPUT)